# Python Itertools and Functools Exercises: 20 Coding Problems with Solutions

A practice notebook on `itertools` (combinatorics, lazy streams, grouping) and `functools` (`reduce`, `partial`, `lru_cache`, `cmp_to_key`) — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-itertools-functools-exercises/). Exercise 16 uses fibonacci(30) instead of the source's fibonacci(35) so the uncached comparison completes quickly in a notebook.*

---

## Concepts you'll need

This set covers Python's **`itertools`** (lazy iterator building blocks) and **`functools`** (higher-order function tools).

**From `itertools`:**
- **Combinatorics** — `permutations(iterable, r)` (order matters), `combinations(iterable, r)` (order doesn't, no repeats), `combinations_with_replacement(iterable, r)` (order doesn't, repeats allowed), `product(*iterables, repeat=n)` (Cartesian product — the tool that replaces nested `for` loops).
- **Combining/repeating streams** — `chain(*iterables)` (concatenate lazily, no copy), `chain.from_iterable(iterable_of_iterables)`, `cycle(iterable)` (repeats forever — pair with `zip()` or `islice()`), `repeat(obj, times=None)` (broadcast a constant, O(1) memory).
- **Infinite sequences, safely bounded** — `count(start, step)` (infinite arithmetic sequence) must be paired with `islice(iterable, stop)` or `takewhile(predicate, iterable)` to avoid running forever.
- **Prefix-based filtering** — `takewhile(pred, it)` stops permanently at the first failure; `dropwhile(pred, it)` skips until the first failure, then yields everything after with no further testing — different from `filter()`, which tests every element independently.
- **`groupby(iterable, key)`** — groups only *consecutive* matching keys, so the data must be **pre-sorted** on that key first, or groups will be wrong. Each group iterator must be materialized (`list(group)`) immediately, before `groupby` advances.
- **`compress(data, selectors)`** — mask-based filtering: keeps elements from `data` wherever the parallel `selectors` value is truthy.
- **`accumulate(iterable, func=add, initial=None)`** — running/cumulative results (sum by default, but any binary function like `operator.mul`, `max`, `min` works).

**From `functools`:**
- **`reduce(function, iterable, initializer=None)`** — folds a sequence down to one value by repeatedly applying a two-argument function; the `operator` module (`operator.add`, `operator.mul`, ...) gives faster, more readable alternatives to simple lambdas.
- **`partial(func, *args, **kwargs)`** — pre-fills some of a function's arguments, returning a new, lower-arity callable — essential for building single-argument predicates for `filter()`/`map()` from multi-argument functions.
- **`lru_cache(maxsize=None)`** (or `@cache`) — memoizes a function's return values by argument, turning exponential-time naive recursion (like Fibonacci) into linear time; `.cache_info()` and `.cache_clear()` inspect/reset it.
- **`cmp_to_key(comparator)`** — bridges an old-style two-argument comparator (returns negative/zero/positive) into a `key=` function `sorted()` can use — needed only when a simple per-element `key` function can't express the ordering logic.

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. All Permutations of a String

**Concept:** itertools.permutations()

**Problem:** Generate all possible arrangements of the characters in a string.

**Given:**
```
chars = "ABC"
```

**Expected Output:**
```
ABC, ACB, BAC, BCA, CAB, CBA — Total: 6
```

**Hint:** permutations(chars) treats elements by position, not value — pass r=2 to restrict output length.

In [ ]:
from itertools import permutations
import math

chars = "ABC"

all_perms = list(permutations(chars))

print(f"All permutations of '{chars}':")
for p in all_perms:
    print(f"  {''.join(p)}")

print(f"\nTotal: {len(all_perms)}")

print(f"\n2-character permutations of '{chars}':")
for p in permutations(chars, 2):
    print(f"  {''.join(p)}")

print(f"\nExpected total (3!): {math.factorial(len(chars))}")

**Explanation:** permutations(iterable, r) returns an iterator of r-length tuples covering every possible ordering; omitting r defaults to the full input length. Each result is a tuple of individual characters like ('A','B','C'), so "".join(p) reassembles it into a readable string. Because permutations() is a generator, it never pre-builds the full list — for a 10-character string it could lazily iterate over 3,628,800 permutations without holding them all in memory; wrapping in list() forces full materialization, fine here but avoided for larger inputs.

## Exercise 2. Combinations Without Replacement

**Concept:** itertools.combinations()

**Problem:** Find all possible 2-player pairs from a team of 5, where order doesn't matter.

**Given:**
```
players = ["Alice", "Bob", "Charlie", "Diana", "Eve"]
```

**Expected Output:**
```
10 unique pairs; Total pairs: 10
```

**Hint:** Unlike permutations(), combinations() never produces both (Alice,Bob) AND (Bob,Alice) — only one ordering.

In [ ]:
from itertools import combinations, permutations
import math

players = ["Alice", "Bob", "Charlie", "Diana", "Eve"]

all_pairs = list(combinations(players, 2))

print(f"All 2-player pairs from {players}:\n")
for i, pair in enumerate(all_pairs, 1):
    print(f"  {i:2d}. {pair[0]} vs {pair[1]}")

print(f"\nTotal pairs: {len(all_pairs)}")
print(f"Verified by math.comb(5,2): {math.comb(5, 2)}")

perm_count = len(list(permutations(players, 2)))
print(f"\nPermutations P(5,2): {perm_count}  (order matters)")
print(f"Combinations C(5,2): {len(all_pairs)}  (order ignored)")
print(f"Ratio P/C = k! = {perm_count // len(all_pairs)}")

**Explanation:** combinations(iterable, r) returns r-length tuples in the input's original relative order, without repetition, treating (Alice,Bob) and (Bob,Alice) as the same pair — only the earlier one is yielded, halving (or more) the output compared to permutations. The count formula C(n,r) = n!/(r!(n-r)!) gives C(5,2)=10; the ratio of permutations to combinations is always r! since each combination can be arranged in r! different orders, which permutations counts separately but combinations collapses into one.

## Exercise 3. Combinations With Replacement

**Concept:** itertools.combinations_with_replacement()

**Problem:** List all possible 2-topping pizza orders, including choosing the same topping twice.

**Given:**
```
toppings = ["Cheese", "Pepperoni", "Mushrooms", "Olives"]
```

**Expected Output:**
```
10 orders including doubles like ('Cheese', 'Cheese')
```

**Hint:** This maintains 'order does not matter' while relaxing the 'no repeats' constraint that plain combinations() has.

In [ ]:
from itertools import combinations_with_replacement, combinations
import math

toppings = ["Cheese", "Pepperoni", "Mushrooms", "Olives"]

orders_with_rep    = list(combinations_with_replacement(toppings, 2))
orders_without_rep = list(combinations(toppings, 2))

print(f"Toppings: {toppings}\n")
print(f"All 2-topping orders (with replacement):")
for i, order in enumerate(orders_with_rep, 1):
    double = " (double topping)" if order[0] == order[1] else ""
    print(f"  {i:2d}. {order[0]} + {order[1]}{double}")

print(f"\nWith replacement    : {len(orders_with_rep)} orders")
print(f"Without replacement : {len(orders_without_rep)} orders")

n, r = len(toppings), 2
print(f"\nFormula C(n+r-1, r) = {math.comb(n + r - 1, r)}")

**Explanation:** combinations_with_replacement() returns r-length tuples chosen from the input where elements CAN repeat, still produced in sorted order relative to the input — so ('Cheese','Cheese') is valid but ('Pepperoni','Cheese') never appears, only ('Cheese','Pepperoni') does. The stars-and-bars count formula C(n+r-1, r) gives C(5,2)=10 for 4 toppings choosing 2: that's 6 unique pairs plus 4 double-topping orders, one per topping paired with itself.

## Exercise 4. Cartesian Product

**Concept:** itertools.product() as a nested-loop replacement

**Problem:** Generate every (size, colour) combination for a product catalogue.

**Given:**
```
sizes = ["S","M","L"], colours = ["Red","Blue","Green","Black"]
```

**Expected Output:**
```
12 (size, colour) pairs; Total variants: 12
```

**Hint:** The rightmost iterable varies fastest, like the innermost loop in a nested for-loop equivalent.

In [ ]:
from itertools import product

sizes   = ["S", "M", "L"]
colours = ["Red", "Blue", "Green", "Black"]

variants = list(product(sizes, colours))

print(f"Sizes   : {sizes}")
print(f"Colours : {colours}")
print(f"\nAll product variants ({len(sizes)} x {len(colours)} = {len(variants)}):")
for i, (size, colour) in enumerate(variants, 1):
    print(f"{i:>4}  {size:<6}  {colour}")

print(f"\nTotal variants: {len(variants)}")

materials = ["Cotton", "Polyester"]
three_attr = list(product(sizes, colours, materials))
print(f"\n3-attribute variants: {len(three_attr)}")

**Explanation:** product(*iterables) computes the Cartesian product, equivalent to nested for loops over each input — the rightmost iterable advances fastest, mirroring the innermost loop's behavior in a manual nested-loop version. Every independent nested for loop can be replaced with a single product() call, flattening indentation and making the combinatorial intent explicit rather than implicit in loop structure. The output size is always multiplicative — 3 sizes x 4 colours = 12 — so adding a third attribute of length 2 doubles the total to 24.

## Exercise 5. Password Generator Using product()

**Concept:** the repeat= parameter for same-alphabet sequences

**Problem:** Generate all possible 4-digit PIN codes using product()'s repeat parameter.

**Given:**
```
digits = "0123456789", pin_length = 4
```

**Expected Output:**
```
Total 4-digit PINs: 10,000
```

**Hint:** product(digits, repeat=4) is cleaner than writing product(digits, digits, digits, digits) manually.

In [ ]:
from itertools import product, islice

digits     = "0123456789"
pin_length = 4

first_5 = ["".join(p) for p in islice(product(digits, repeat=pin_length), 5)]
print(f"First 5 PINs : {first_5}")

total = len(digits) ** pin_length
print(f"Total {pin_length}-digit PINs: {total:,}")

print(f"\nSearch space growth (alphabet size = {len(digits)}):")
prev = 1
for length in range(1, 7):
    count = len(digits) ** length
    factor = count // prev
    print(f"  length {length}: {count:>10,} PINs  ({factor}x previous)")
    prev = count

**Explanation:** repeat=n is shorthand for passing the same iterable n times — product("AB", repeat=3) is identical to product("AB","AB","AB"), producing all 8 three-character binary strings, and is the correct tool for fixed-length sequences from a single alphabet. islice(iterator, n) consumes only the first n elements without forcing the rest to generate — essential for previewing when the full space (here 10,000, but a 10-character password space would be 10 billion) is too large to fully materialize. Each additional character multiplies the search space by the alphabet size, which is exactly why longer passwords are dramatically harder to brute-force.

## Exercise 6. Chain Multiple Iterables

**Concept:** itertools.chain() for lazy, copy-free merging

**Problem:** Merge three department employee lists into one iterable without an intermediate combined list.

**Given:**
```
engineering, marketing, hr — three separate lists
```

**Expected Output:**
```
All 8 names printed in order; Total employees: 8
```

**Hint:** chain.from_iterable() is the version to use when the lists are stored inside a container rather than known individually.

In [ ]:
from itertools import chain

engineering = ["Alice", "Bob", "Charlie"]
marketing   = ["Diana", "Eve"]
hr          = ["Frank", "Grace", "Heidi"]

all_employees = list(chain(engineering, marketing, hr))

print("All employees (chained):")
for name in all_employees:
    print(f"  {name}")

print(f"\nTotal employees: {len(all_employees)}")

departments = [engineering, marketing, hr]
all_from_iterable = list(chain.from_iterable(departments))
print(f"\nUsing chain.from_iterable: {all_from_iterable}")
print(f"Results match: {all_employees == all_from_iterable}")

**Explanation:** chain(*iterables) returns a single iterator that reads from the first source until exhausted, then the second, and so on — no new list is allocated, the originals are simply read through in sequence, making it more memory-efficient than the + operator for large collections. chain.from_iterable() is the alternative constructor for when the sources are already grouped inside a container (like a list of lists) rather than passed individually. A chain() iterator is single-use: once fully consumed, further iteration yields nothing, so materializing to a list up front is needed if you'll need the merged data more than once.

## Exercise 7. Infinite Counter with islice

**Concept:** itertools.count() paired with islice() for safe bounded consumption

**Problem:** Create an infinite counter and safely extract a fixed number of values from it.

**Given:**
```
start = 100, step = 5, extract 8 values
```

**Expected Output:**
```
[100, 105, 110, 115, 120, 125, 130, 135]
```

**Hint:** Never iterate over count() directly with a bare for loop — always pair it with islice() or takewhile().

In [ ]:
from itertools import count, islice, takewhile

counter = count(start=100, step=5)
first_8 = list(islice(counter, 8))
print(f"First 8 values (start=100, step=5): {first_8}")

counter2  = count(start=0, step=1)
every_3rd = list(islice(counter2, 0, 15, 3))
print(f"Every 3rd from 0 (islice step=3)  : {every_3rd}")

counter3  = count(start=100, step=5)
below_140 = list(takewhile(lambda x: x < 140, counter3))
print(f"Values below 140                   : {below_140}")

id_gen = count(start=1001)
new_ids = [next(id_gen) for _ in range(5)]
print(f"\nGenerated IDs : {new_ids}")

**Explanation:** count(start, step) produces an infinite evenly-spaced sequence with no stop argument at all, unlike range(); it also supports float steps cleanly, without the accumulation error repeated addition would introduce. islice(iterable, stop) consumes at most stop elements and then stops reading entirely — the full islice(iterable, start, stop, step) form mirrors Python's slice syntax but works on ANY iterator, including infinite ones, making it the safest general way to cap an unbounded source. next(id_gen) advances a count() object by one step on demand, a pattern used to generate monotonically increasing IDs similar to a database auto-increment sequence.

## Exercise 8. Cycle Through a List

**Concept:** itertools.cycle() for automatic wrap-around, paired with zip()

**Problem:** Rotate through shift labels and assign one to each of 10 employees, wrapping around as needed.

**Given:**
```
shifts = ["Morning","Afternoon","Night"], 10 employees
```

**Expected Output:**
```
Each employee assigned a shift, cycling and repeating
```

**Hint:** zip() pairing a finite list with an infinite cycle() is the idiomatic way to safely consume the cycle.

In [ ]:
from itertools import cycle, islice

shifts    = ["Morning", "Afternoon", "Night"]
employees = ["Alice", "Bob", "Charlie", "Diana", "Eve",
             "Frank", "Grace", "Heidi", "Ivan", "Judy"]

schedule = list(zip(employees, cycle(shifts)))
print("Shift Schedule:")
for employee, shift in schedule:
    print(f"  {employee:<10}  {shift}")

n_items = 7
cycled_labels = list(islice(cycle(shifts), n_items))
print(f"\nFirst {n_items} shift labels (islice): {cycled_labels}")

**Explanation:** cycle(iterable) returns an infinite iterator repeatedly yielding the input's elements, restarting from the beginning every time it reaches the end — it stores a copy of the input internally to replay it, so unlike count() it uses memory proportional to the input's length. zip(employees, cycle(shifts)) is the idiomatic way to safely consume an infinite cycle: zip() stops the moment its SHORTEST input (the finite employees list) is exhausted, so the infinite cycle is never read past what's actually needed. Without cycle(), the same result would need shifts[i % len(shifts)] inside a manual loop — cycle() eliminates that index arithmetic entirely and composes cleanly with islice() and zip().

## Exercise 9. Repeat a Value

**Concept:** itertools.repeat() for broadcasting a constant with O(1) memory

**Problem:** Pair a fixed discount rate with every item in a product list to compute sale prices.

**Given:**
```
products with prices, discount_rate = 0.15
```

**Expected Output:**
```
Each product shown with original, discount, and sale price
```

**Hint:** repeat(x) yields references to the same object indefinitely — far cheaper than a list comprehension like [rate]*n.

In [ ]:
from itertools import repeat

products = [
    ("Laptop",   999.99),
    ("Mouse",     29.99),
    ("Keyboard",  79.99),
    ("Monitor",  349.99),
]
discount_rate = 0.15

print(f"Discount rate: {discount_rate * 100:.0f}%\n")
for (name, price), rate in zip(products, repeat(discount_rate)):
    discount   = price * rate
    sale_price = price - discount
    print(f"  {name:<10}  Orig: {price:>8.2f}  Disc: {discount:>7.2f}  Sale: {sale_price:>8.2f}")

import math
prices = [p for _, p in products]
log_prices = list(map(math.log, prices, repeat(10)))
print(f"\nlog10 of each price:")
for (name, _), lp in zip(products, log_prices):
    print(f"  {name:<10}: {lp:.4f}")

**Explanation:** repeat(object, times=None) yields the same object indefinitely when times is omitted, or exactly times times when given — and because it hands back a reference rather than copies, it uses O(1) memory regardless of the count, far more efficient than a list comprehension like [rate] * n. map(math.log, prices, repeat(10)) computes math.log(price, 10) for every price without a lambda, applying a two-argument function where one argument stays fixed across the whole sequence. An infinite repeat(x) must be paired with a finite iterable (via zip()) or capped with islice(); the finite repeat(x, n) form is safe to materialize directly whenever the count is known upfront.

## Exercise 10. takewhile and dropwhile

**Concept:** prefix-based filtering, contrasted with filter()

**Problem:** Collect temperature readings while below a threshold, then separately skip everything below it.

**Given:**
```
temperatures = [18.2, 21.5, 24.8, 27.1, 29.6, 31.3, 28.4, 33.7, 30.1, 26.5]
```

**Expected Output:**
```
takewhile stops at 31.3; dropwhile includes everything from 31.3 onward, even the later 26.5
```

**Hint:** takewhile stops PERMANENTLY at the first failure — it never rechecks later elements, unlike filter().

In [ ]:
from itertools import takewhile, dropwhile, chain

temperatures = [18.2, 21.5, 24.8, 27.1, 29.6, 31.3, 28.4, 33.7, 30.1, 26.5]
threshold    = 30.0

below_30 = list(takewhile(lambda x: x < threshold, temperatures))
from_hot = list(dropwhile(lambda x: x < threshold, temperatures))

print(f"Full sequence               : {temperatures}")
print(f"\ntakewhile (below {threshold})   : {below_30}")
print(f"dropwhile (skip below {threshold}) : {from_hot}")

recovered = list(chain(below_30, from_hot))
print(f"\nchain of both halves recovers original: {recovered == temperatures}")

filtered = list(filter(lambda x: x < threshold, temperatures))
print(f"\nfilter(< {threshold})           : {filtered}")
print("Note: filter() finds 26.5 at the end; takewhile() stops at 31.3 and misses it.")

**Explanation:** takewhile(predicate, iterable) yields elements only as long as predicate is True — the moment it returns False for ANY element, iteration stops immediately and permanently, even if later elements would have passed. dropwhile(predicate, iterable) skips elements while predicate is True, and once it first returns False, yields every remaining element with no further testing — which is why 26.5 (itself below 30) still appears at the end, since the drop phase already ended at 31.3. This is fundamentally different from filter(), which independently tests every single element. Together, takewhile and dropwhile with the same predicate split a sequence at its first predicate-breaking point — chaining their two outputs always reconstructs the original exactly.

## Exercise 11. Group Consecutive Items

**Concept:** itertools.groupby() — requires pre-sorting on the grouping key

**Problem:** Group a list of transactions by category and compute per-category subtotals.

**Given:**
```
7 transactions across Food/Travel/Tech categories, unsorted
```

**Expected Output:**
```
Each category with its item count and subtotal
```

**Hint:** groupby() ONLY groups consecutive identical keys — sort by the key first, or groups will be wrong.

In [ ]:
from itertools import groupby

transactions = [
    ("Food",   12.50), ("Food",    8.30), ("Travel", 45.00),
    ("Travel", 22.10), ("Travel", 15.75), ("Food",    6.90),
    ("Tech",  299.99),
]

sorted_txns = sorted(transactions, key=lambda x: x[0])

print("Transactions grouped by category:\n")
for category, group in groupby(sorted_txns, key=lambda x: x[0]):
    items    = list(group)
    subtotal = sum(amount for _, amount in items)
    print(f"  {category} ({len(items)} items, subtotal: {subtotal:.2f})")
    for _, amount in items:
        print(f"    {amount:.2f}")

print("\nWithout sorting (incorrect grouping):")
for category, group in groupby(transactions, key=lambda x: x[0]):
    items = list(group)
    print(f"  {category}: {len(items)} item(s)")

**Explanation:** groupby(iterable, key) returns consecutive (key, group_iterator) pairs, grouping only elements that are ADJACENT and share the same key — it never looks ahead or sorts on your behalf, so pre-sorting by the grouping key is mandatory. Materializing each group with list(group) IMMEDIATELY inside the loop is essential: the group iterator shares an internal pointer with the parent, and it's silently exhausted the moment groupby() advances to the next key. The unsorted demonstration shows 'Food' splitting into two separate groups because 'Travel' interrupts it — this produces no error, just silently wrong groups, which is exactly why the pre-sort requirement matters.

## Exercise 12. Compress a Sequence

**Concept:** itertools.compress() for mask-based (boolean-selector) filtering

**Problem:** Filter a product list using a parallel boolean/int selector list.

**Given:**
```
products list, in_stock = [1, 0, 1, 1, 0, 1]
```

**Expected Output:**
```
Available products: ['Laptop', 'Keyboard', 'Monitor', 'Webcam']
```

**Hint:** compress() stops at the end of the SHORTER of the two inputs — data or selectors — whichever runs out first.

In [ ]:
from itertools import compress

products  = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones", "Webcam"]
in_stock  = [1, 0, 1, 1, 0, 1]

available = list(compress(products, in_stock))
print(f"All products  : {products}")
print(f"In stock mask : {in_stock}")
print(f"Available     : {available}")

out_of_stock = list(compress(products, [not s for s in in_stock]))
print(f"Out of stock  : {out_of_stock}")

products_with_prices = [
    ("Laptop", 999.99), ("Mouse", 29.99), ("Keyboard", 79.99),
    ("Monitor", 349.99), ("Headphones", 49.99), ("Webcam", 89.99),
]
threshold = 75.0
selector  = [price > threshold for _, price in products_with_prices]
names     = [name for name, _ in products_with_prices]

premium = list(compress(names, selector))
print(f"\nProducts above {threshold}: {premium}")

**Explanation:** compress(data, selectors) yields elements from data at positions where the corresponding selectors value is truthy, stopping as soon as either input runs out; selector values can be any truthy-evaluable type — ints, booleans, or objects. Negating the selector with [not s for s in in_stock] gives the complement (out-of-stock items), mirroring how NumPy's boolean mask negation (~mask) works — making compress() the pure-Python equivalent of mask-based indexing. The selector doesn't need to be a static hardcoded list at all; it can be computed dynamically at runtime from the data itself, as the price-threshold example shows.

## Exercise 13. Accumulate Running Totals

**Concept:** itertools.accumulate() with custom binary functions

**Problem:** Compute running total, running max, and running min of monthly sales in one pass each.

**Given:**
```
monthly_sales = [12000, 15400, 9800, 18200, 21000, 14300, 16700]
```

**Expected Output:**
```
Running total/max/min printed alongside each month
```

**Hint:** accumulate(data, max) works because Python's built-in max(a,b) is itself a valid two-argument function.

In [ ]:
from itertools import accumulate
import operator

monthly_sales = [12000, 15400, 9800, 18200, 21000, 14300, 16700]
months        = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul"]

running_total = list(accumulate(monthly_sales))
running_max   = list(accumulate(monthly_sales, max))
running_min   = list(accumulate(monthly_sales, min))

for month, sales, total, rmax, rmin in zip(months, monthly_sales, running_total, running_max, running_min):
    print(f"{month}: sales={sales:>6,}  cumulative={total:>6,}  max={rmax:>6,}  min={rmin:>6,}")

growth_factors = [1.05, 0.98, 1.12, 1.03, 0.97, 1.08, 1.01]
running_product = list(accumulate(growth_factors, operator.mul))
print(f"\nRunning compound growth:")
for month, factor, product in zip(months, growth_factors, running_product):
    print(f"  {month}: factor={factor:.2f}  cumulative={product:.4f}")

with_initial = list(accumulate(monthly_sales[:3], initial=0))
print(f"\nWith initial=0: {with_initial}")

**Explanation:** accumulate(iterable, func=operator.add, initial=None) applies a two-argument function cumulatively, yielding EVERY intermediate result, not just the final one — the default function is addition, giving a running sum. Because built-in max(a,b) and min(a,b) each accept exactly two arguments, they slot directly in as the accumulation function with zero helper code needed. The initial keyword (Python 3.8+) prepends that value as the very first output before any real element is processed, which shifts the output length to len(data) + 1 — useful when the accumulation needs a defined starting state, like an account balance beginning at 0.

## Exercise 14. Reduce to a Single Value

**Concept:** functools.reduce() as the general-purpose fold operation

**Problem:** Compute the product of a list using reduce(), then extend it to max, string concat, and flattening.

**Given:**
```
numbers = [2, 3, 4, 5, 6]
```

**Expected Output:**
```
Product: 720 (matching math.prod)
```

**Hint:** operator.mul is a faster, more readable substitute for lambda a, b: a * b.

In [ ]:
from functools import reduce
import operator
import math

numbers = [2, 3, 4, 5, 6]

product = reduce(operator.mul, numbers)
print(f"Numbers          : {numbers}")
print(f"Product (reduce) : {product}")
print(f"math.prod verify : {math.prod(numbers)}")

maximum = reduce(lambda a, b: a if a > b else b, numbers)
print(f"\nMaximum (reduce) : {maximum}")

words = ["Python", " is", " powerful"]
sentence = reduce(lambda a, b: a + b, words)
print(f"\nConcatenated     : '{sentence}'")

nested = [[1, 2], [3, 4], [5, 6], [7]]
flat = reduce(lambda acc, lst: acc + lst, nested, [])
print(f"\nFlattened        : {flat}")

empty_product = reduce(operator.mul, [], 1)
print(f"\nreduce on empty list (initializer=1): {empty_product}")

**Explanation:** reduce(function, iterable, initializer) applies function to the first two elements, then to that result and the third element, and so on until one value remains; the optional initializer sits before the iterable's first element and is returned unchanged if the iterable is empty, making the call safe against empty input. operator.mul is a pre-compiled C-level function performing multiplication — faster AND more readable than an equivalent lambda a, b: a * b, and the operator module has an equivalent for essentially every Python operator. Prefer built-ins (sum(), max(), math.prod()) when they exist; reach for reduce() for custom aggregations with no built-in equivalent, like flattening nested lists or building a running set intersection.

## Exercise 15. Partial Functions

**Concept:** functools.partial() for pre-filling arguments

**Problem:** Create a specialized power_of_2() function from the built-in pow() using partial().

**Given:**
```
exponents = [0, 1, 2, ..., 10]
```

**Expected Output:**
```
[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
```

**Hint:** partial(pow, 2) pre-fills pow()'s first argument, leaving only the exponent to be supplied later.

In [ ]:
from functools import partial

exponents = list(range(11))

power_of_2 = partial(pow, 2)
power_of_3 = partial(pow, 3)

powers_of_2 = list(map(power_of_2, exponents))
powers_of_3 = list(map(power_of_3, exponents))

print(f"Exponents   : {exponents}")
print(f"Powers of 2 : {powers_of_2}")
print(f"Powers of 3 : {powers_of_3}")

def greet(name, greeting="Hello", punctuation="!"):
    return f"{greeting}, {name}{punctuation}"

formal_greet = partial(greet, greeting="Good morning", punctuation=".")
casual_greet = partial(greet, greeting="Hey")

names = ["Alice", "Bob", "Charlie"]
print(f"\nFormal greetings:")
for name in names:
    print(f"  {formal_greet(name)}")

print(f"\npartial object info:")
print(f"  power_of_2.func   : {power_of_2.func}")
print(f"  power_of_2.args   : {power_of_2.args}")

**Explanation:** partial(func, *args, **kwargs) returns a new callable with the given arguments pre-filled; when that partial is later called, the pre-filled arguments are prepended to whatever additional arguments are supplied at call time, behaving exactly like calling the original function with everything combined. A partial object exposes .func (the wrapped function), .args (pre-filled positional arguments), and .keywords (pre-filled keyword arguments) — useful for debugging or introspecting callables. partial is preferred over an equivalent lambda when the function and its fixed arguments are all that's needed: it's more explicit, introspectable, and avoids the late-binding closure pitfalls lambdas can introduce inside loop bodies.

## Exercise 16. Memoisation with lru_cache

**Concept:** functools.lru_cache() turning exponential recursion into linear

**Problem:** Compare cached vs. uncached recursive Fibonacci for both speed and call count.

**Given:**
```
fibonacci(30) (reduced from the source's 35 for faster notebook execution)
```

**Expected Output:**
```
Both cached and uncached agree on the result; cache misses equal n+1 unique subproblems
```

**Hint:** @lru_cache(maxsize=None) creates an unbounded cache; .cache_info() reports hits/misses/size directly.

In [ ]:
from functools import lru_cache
import time

call_count_uncached = 0
def fib_uncached(n):
    global call_count_uncached
    call_count_uncached += 1
    if n <= 1:
        return n
    return fib_uncached(n - 1) + fib_uncached(n - 2)

@lru_cache(maxsize=None)
def fib_cached(n):
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

n = 30

start = time.perf_counter()
result_uncached = fib_uncached(n)
time_uncached   = time.perf_counter() - start

start = time.perf_counter()
result_cached = fib_cached(n)
time_cached   = time.perf_counter() - start

print(f"fibonacci({n}) = {result_cached}")
print(f"Results match: {result_uncached == result_cached}")

print(f"\nUncached: time={time_uncached:.4f}s  calls={call_count_uncached:,}")

info = fib_cached.cache_info()
print(f"Cached  : time={time_cached:.6f}s  hits={info.hits}  misses={info.misses}  size={info.currsize}")

speedup = time_uncached / time_cached if time_cached > 0 else float("inf")
print(f"\nSpeedup: ~{speedup:.0f}x")

fib_cached.cache_clear()
print(f"After cache_clear(): {fib_cached.cache_info()}")

**Explanation:** @lru_cache(maxsize=None) stores a function's return values in a dictionary keyed by its arguments, so a later call with identical arguments returns the cached value instantly instead of re-executing the body; maxsize=None makes the cache unbounded (Python 3.9+'s plain @cache decorator is an equivalent shorthand). The uncached version recomputes every overlapping subproblem from scratch, making millions of calls, while the cached version makes exactly one call per unique n value (0 through n) — collapsing the time complexity from O(2^n) down to O(n). cache_info() reports hits (served from cache), misses (actually computed), and currsize; cache_clear() empties the cache entirely, useful when cached data might have gone stale.

## Exercise 17. Custom Sorting with cmp_to_key

**Concept:** functools.cmp_to_key() to bridge a comparator into sorted()

**Problem:** Sort software version strings numerically, not lexicographically.

**Given:**
```
versions = ["1.10.2", "1.9.1", "2.0.0", "1.10.1", "1.2.3", "2.1.0", "1.10.10"]
```

**Expected Output:**
```
1.2.3, 1.9.1, 1.10.1, 1.10.2, 1.10.10, 2.0.0, 2.1.0
```

**Hint:** Lexicographic sort fails for versions since string comparison is character-by-character: '10' < '9' as strings.

In [ ]:
from functools import cmp_to_key

versions = ["1.10.2", "1.9.1", "2.0.0", "1.10.1",
            "1.2.3",  "2.1.0", "1.10.10"]

lex_sorted = sorted(versions)
print(f"Lexicographic (wrong) : {lex_sorted}")

def version_comparator(v1, v2):
    parts1 = [int(x) for x in v1.split(".")]
    parts2 = [int(x) for x in v2.split(".")]
    if parts1 < parts2:
        return -1
    if parts1 > parts2:
        return 1
    return 0

correct_sorted = sorted(versions, key=cmp_to_key(version_comparator))
print(f"Numeric (correct)     : {correct_sorted}")

def version_key(v):
    return tuple(int(x) for x in v.split("."))

key_sorted = sorted(versions, key=version_key)
print(f"\nKey function (simpler): {key_sorted}")
print(f"Both approaches match : {correct_sorted == key_sorted}")

**Explanation:** cmp_to_key(comparator) wraps a two-argument comparator function (returning negative/zero/positive) into a key class supporting <, >, == — the compatibility bridge for pairwise ordering logic, since Python 3's sorted() removed direct comparator support entirely. String comparison is purely character-by-character, so '1.10.2' sorts before '1.9.1' lexicographically because the character '1' is less than '9' — converting each version's dot-separated parts to integers fixes this, since the tuple (1,10,2) correctly sorts after (1,9,1). When ordering can be expressed by transforming each element independently (as version tuples can), a plain key function is simpler and faster — Python calls it once per element, versus a comparator potentially called O(n log n) times; reserve cmp_to_key() for genuinely cross-element comparison logic like locale-aware sorting.

## Exercise 18. Pipeline with chain and reduce

**Concept:** combining lazy merging (chain) with folding (reduce) — a mini ETL pattern

**Problem:** Merge regional sales data and compute total/max/min revenue in a clean pipeline.

**Given:**
```
north, south, east — three regional sales lists
```

**Expected Output:**
```
Total, maximum, and minimum revenue across all regions combined
```

**Hint:** Materialize the chained stream to a list ONCE if you need it for multiple separate aggregations.

In [ ]:
from itertools import chain
from functools import reduce
import operator

north = [15200, 18400, 12100]
south = [22300, 19800, 25100]
east  = [11500, 14200, 16800, 13400]

all_sales = list(chain(north, south, east))
n = len(all_sales)

print(f"Combined: {all_sales}")

total   = reduce(operator.add, all_sales)
maximum = reduce(lambda a, b: a if a > b else b, all_sales)
minimum = reduce(lambda a, b: a if a < b else b, all_sales)
average = total / n

print(f"\nTotal revenue: {total:,}")
print(f"Maximum sale : {maximum:,}")
print(f"Minimum sale : {minimum:,}")
print(f"Average sale : {average:,.2f}")

def combined_reducer(acc, x):
    total, mx, mn = acc
    return (total + x, max(mx, x), min(mn, x))

first = all_sales[0]
total2, max2, min2 = reduce(combined_reducer, all_sales[1:], (first, first, first))
print(f"\nSingle-pass reduce: total={total2:,}  max={max2:,}  min={min2:,}")
print(f"Matches separate passes: {total == total2 and maximum == max2 and minimum == min2}")

**Explanation:** chain() returns a one-use iterator; if the same merged data is needed for multiple separate aggregations, converting it to a list ONCE with list(chain(...)) avoids re-creating the chain repeatedly and prevents the silent empty-result bug of iterating an already-spent iterator a second time. Rather than three separate reduce() passes, a single pass can accumulate a state tuple (total, max, min) simultaneously, touching each element exactly once — for very large datasets this cuts the number of iterations by two-thirds. This chain-then-reduce combination mirrors the Map-Reduce pattern directly: chain() is the gather/extract step, reduce() is the fold/aggregate step, with map() or a generator expression able to sit between them for a fuller transform stage.

## Exercise 19. Cached Combination Counter

**Concept:** lru_cache applied to Pascal's triangle recursion

**Problem:** Implement C(n,k) via Pascal's recurrence with memoization, comparing call counts to the uncached version.

**Given:**
```
comb(10,3), comb(15,6), comb(20,10)
```

**Expected Output:**
```
Cached results match math.comb(); cache misses far below uncached call counts
```

**Hint:** C(n,k) = C(n-1,k-1) + C(n-1,k), base cases C(n,0)=1 and C(n,n)=1 — massive overlap in the naive recursion tree.

In [ ]:
from functools import lru_cache
import math

uncached_calls = 0
def comb_uncached(n, k):
    global uncached_calls
    uncached_calls += 1
    if k == 0 or k == n:
        return 1
    return comb_uncached(n - 1, k - 1) + comb_uncached(n - 1, k)

@lru_cache(maxsize=None)
def comb_cached(n, k):
    if k == 0 or k == n:
        return 1
    return comb_cached(n - 1, k - 1) + comb_cached(n - 1, k)

test_cases = [(10, 3), (15, 6), (20, 10)]
for n, k in test_cases:
    cached  = comb_cached(n, k)
    builtin = math.comb(n, k)
    print(f"C({n},{k}) = {cached:,}  (math.comb: {builtin:,}, match: {cached == builtin})")

info = comb_cached.cache_info()
print(f"\nCache: hits={info.hits}  misses={info.misses} (unique subproblems)  size={info.currsize}")

uncached_calls = 0
comb_uncached(20, 10)
print(f"\nC(20,10) uncached calls : {uncached_calls:,}")
print(f"C(20,10) cached misses  : {info.misses}")

**Explanation:** Pascal's recurrence C(n,k) = C(n-1,k-1) + C(n-1,k) reflects choosing whether a specific item is included (first term) or excluded (second term); without caching, this recursive tree has massive overlap — the same (n,k) subproblem gets recomputed independently down many different branches. Each unique (n,k) pair is computed exactly once (a cache miss) and served from cache on every later call (a hit); for C(20,10), only 231 unique subproblems exist total, while the uncached version makes tens of thousands of redundant calls. Because lru_cache persists across separate calls to the same function, subproblems computed while solving comb_cached(10,3) are automatically reused later while solving comb_cached(20,10) — the cache accumulates value over the program's whole lifetime.

## Exercise 20. Lazy Data Pipeline with islice, filter, and partial

**Concept:** composing itertools + functools into a full lazy ETL pipeline

**Problem:** Build a lazy pipeline: cap a stream to 200 records, filter by region+amount, then aggregate.

**Given:**
```
a simulated stream of sales records as (product, region, amount) tuples
```

**Expected Output:**
```
Total revenue from qualifying North-region, high-value sales in the first 200 records
```

**Hint:** partial() turns a 3-argument predicate into the 1-argument callable filter() requires.

In [ ]:
from itertools import islice
from functools import reduce, partial
import random

random.seed(42)

regions  = ["North", "South", "East", "West"]
products = ["Widget", "Gadget", "Doohickey", "Thingamajig"]

def sales_stream(n):
    for _ in range(n):
        yield (
            random.choice(products),
            random.choice(regions),
            round(random.uniform(50, 2000), 2),
        )

def is_qualifying(record, region, min_amount):
    _, rec_region, amount = record
    return rec_region == region and amount >= min_amount

north_high_value = partial(is_qualifying, region="North", min_amount=500)

stream     = sales_stream(1000)
capped     = islice(stream, 200)
qualifying = filter(north_high_value, capped)
results    = list(qualifying)

if results:
    total   = reduce(lambda acc, r: acc + r[2], results, 0.0)
    maximum = reduce(lambda a, b: a if a[2] > b[2] else b, results)
    count   = len(results)
else:
    total, count = 0.0, 0

print("Pipeline: islice(200) -> filter(North, >=500) -> reduce")
print(f"\nQualifying records  : {count}")
print(f"Total revenue       : {total:,.2f}")
if results:
    print(f"Highest single sale : {maximum[2]:,.2f} ({maximum[0]}, {maximum[1]})")

print(f"\nSample qualifying records (first 5):")
for product, region, amount in results[:5]:
    print(f"  {product:<15}  {region:<6}  {amount:,.2f}")

**Explanation:** Each stage — islice(), filter() — returns an iterator rather than a materialized list, so no records are pulled from the generator until the final list() call forces the whole pipeline to run; a pipeline over a million records here still only reads the 200 that islice() actually requests, keeping memory flat regardless of stream size. filter() requires a single-argument predicate, but is_qualifying() needs three parameters — partial(is_qualifying, region="North", min_amount=500) pre-fills the configuration and yields a single-argument function filter() can call directly, avoiding a lambda while keeping the logic in a named, independently testable function. The pipeline stays lazy right up until list(qualifying) forces evaluation in one coordinated pass — using reduce() directly on the filter iterator instead would skip even that final list allocation, at the cost of losing the ability to inspect results or run multiple aggregations without re-running the whole pipeline.